In [1]:
import os
os.chdir('..')
print(os.getcwd())

d:\pythonProject\IC Lab\Gait_analysis\pyskl


In [2]:
from colab.tools import prinfo
import torch

In [3]:
split = 8
X = (3.0, 157.0, 1.0, 3.0, 0.0, 0.0, 25.0, 27.0, 66.0, 5.0, 90.0, 66.0, 50.0)

In [4]:
params_range = {
    'num_init'      : (1, 4),
    'base_channel'  : (64, 256),
    'stride_init'   : (1, 3),
    'tkernel_init'  : [3, 5, 7],
    
    'act'           : (0, 3),           # activate function  
    'opt'           : (0, 2),           # optimizer

    'dropout_bk'    : (15, 30),         # dropout in block *10-2
    'dropout_fc'    : (15, 30),         # dropout in fc *10-2

    'lr'            : (1, 100),         # learning rate *10-3
    'weight_decay'  : (0, 100),         # weight_decay *10-4
    'momentum'      : (50, 99),          # momentum * 10-2
    'margin'        : (20, 100),          # margin in Triplet Loss *10-2
    'lambda_val'    : (0, 90),          # The ratio of Triplet Loss compare to Cross Entropy *10-2
}

# 設定初始解
params = {
    'data'          : 'drunk',          # 我自己讀特定資料集的方式
    'batch'         : 128,              # 可以調整~~
    'epoch'         : 80,               # 可以調整~~
    'feature'       : 'j',              # 我自己讀特定資料集的方式
    'split'         : split,
    
    'num_init'      : 4,
    'base_channel'  : 64,           # output_channel in init_layer
    'stride_init'   : 1,
    'tkernel_init'  : 3,

    'num_in'        : 0,                # num_layers_input_stream 固定捨棄後面兩個stream
    'num_main'      : 0,                # num_layers_main_stream 固定捨棄後面兩個stream

    'act'           : 0,                # activate function
    'opt'           : 0,                # optimizer
    
    'dropout_bk'    : 0,                # dropout in block
    'dropout_fc'    : 0,                # dropout in fc

    'lr'            : 100,              # learning rate
    'weight_decay'  : 5,                # weight_decay
    'momentum'      : 90,               # momentum
    'margin'        : 0.6,              # margin in Triplet Loss
    'lambda_val'    : 0.5,              # The ratio of Triplet Loss compare to Cross Entropy
}
                                                                                                        
# 不用動
def get_param(X, keys=params_range.keys(), params=params):
    """
    將搜尋空間中編碼過的數值 X 映射回實際的模型參數設定，統一保留4位小數。

    Parameters:
        X (np.ndarray): 搜尋演算法產出的參數值列表。
        keys (list): 與 X 對應的參數名稱。
        params (dict): 預設參數字典。

    Returns:
        dict: 還原為實際值後的參數字典。
    """
    param = params.copy()

    for i, key in enumerate(keys):
        val = X[i]
        boundary = params_range[key]

        # 統一保留 4 位小數的縮放邏輯
        if key in ['dropout_bk', 'dropout_fc']:
            param[key] = round(val * 1e-2, 4)
        elif key == 'lr':
            param[key] = round(val * 1e-3, 4)
        elif key == 'weight_decay':
            param[key] = round(val * 1e-4, 4)
        elif key == 'momentum':
            param[key] = round(val * 1e-2, 4)
        elif key == 'margin':
            param[key] = round(val * 1e-2, 4)
        elif key == 'lambda_val':
            param[key] = round(val * 1e-2, 4)
        elif (
            isinstance(boundary, (tuple, list)) and
            all(isinstance(b, int) for b in boundary)
        ):
            param[key] = int(val)
        else:
            param[key] = val

    return param

def fitness(X, run=None):
    # **釋放 GPU 記憶體**
    torch.cuda.empty_cache()
    
    param = get_param(X=X)
    param_args = " ".join([f"--{key} {str(value).strip()}" for key, value in param.items()])
    command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu --mt {param_args}"
    
    try:
        # 使用 `!{command}` 在 Jupyter Notebook / Colab 內執行
        output = !{command}
        # print(output)
        # 解析輸出
        train_cost, train_time, test_time, val_acc_4c, val_acc_3c, val_acc_2c, \
        test_acc_4c, test_acc_3c, test_acc_2c, f1, pre, rec, log = prinfo(output)
        
        if run is not None:
            print(f"Run : {run:>3} | Train_cost : {train_cost:>8.3f}s | Train_time : {train_time:>8.7f}s | "
                  f"Test_time : {test_time:>8.3f}s | Val_acc_4c : {val_acc_4c:>10.7f} | "
                  f"Val_acc_3c : {val_acc_3c:>10.7f} | Val_acc_2c : {val_acc_2c:>10.7f} | "
                  f"Test_acc_4c : {test_acc_4c:>10.7f} | Test_acc_3c : {test_acc_3c:>10.7f} | "
                  f"Test_acc_2c : {test_acc_2c:>10.7f} | F1 : {f1:>10.7f} | "
                  f"Precision : {pre:>10.7f} | Recall : {rec:>10.7f} | Log : {log}")

    except Exception as e:
        print(f"❌ 發生錯誤: {e}")
        # 確保所有變數有值，避免 `NoneType` 出錯
        train_cost = train_time = test_time = 0
        val_acc_4c = val_acc_3c = val_acc_2c = 0
        test_acc_4c = test_acc_3c = test_acc_2c = 0
        f1 = pre = rec = 0
        log = ""

    # 確保 `NoneType` 不影響計算
    fitness_value = (val_acc_4c or 0)

    # 確保 `log` 不為 `None`
    log = log if log is not None else ""

    record_message = {
        'train_cost': train_cost,
        'train_time': train_time,
        'test_time': test_time,
        'val_acc_4c': val_acc_4c,
        'val_acc_3c': val_acc_3c,
        'val_acc_2c': val_acc_2c,
        'test_acc_4c': test_acc_4c,
        'test_acc_3c': test_acc_3c,
        'test_acc_2c': test_acc_2c,
        'f1': f1,
        'precision': pre,
        'recall': rec,
        'log': log
    }
    return fitness_value, record_message

In [5]:
split = 8
X = (3.0, 157.0, 1.0, 3.0, 0.0, 0.0, 25.0, 27.0, 66.0, 5.0, 90.0, 66.0, 50.0)
for run in range(1):
    fitness_value, record_message = fitness(X=X, run=run)

Run :   0 | Train_cost :  546.561s | Train_time : 518.7829609s | Test_time :    1.632s | Val_acc_4c :  0.8271484 | Val_acc_3c :  0.9335938 | Val_acc_2c :  0.9941406 | Test_acc_4c :  0.7861111 | Test_acc_3c :  0.8694444 | Test_acc_2c :  0.9847222 | F1 :  0.9845506 | Precision :  0.9957386 | Recall :  0.9736111 | Log : results/record_20250424_170328_drunk_j_split8_4C-0.7861_3C-0.8694_2C-0.9847_2CP-0.9957_2CR-0.9736_2CF-0.9846.txt
